In [1]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import matplotlib
matplotlib.use("Agg")

In [2]:
import torch
import requests
import numpy as np
import cv2
import matplotlib.pyplot as plt

from PIL import Image

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

from segment_anything import (
    sam_model_registry,
    SamPredictor
)

C:\Users\Admin\anaconda3\envs\ovs-thesis\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Using device: cuda
NVIDIA RTX PRO 4000 Blackwell


In [4]:
model_id = "IDEA-Research/grounding-dino-tiny"

processor = AutoProcessor.from_pretrained(
    model_id
)

grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    model_id
).to(DEVICE)

print("GroundingDINO loaded")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 990/990 [00:00<00:00, 8667.91it/s]


GroundingDINO loaded


In [5]:
SAM_CHECKPOINT = "../checkpoints/sam_vit_h_4b8939.pth"

sam = sam_model_registry["vit_h"](
    checkpoint=SAM_CHECKPOINT
)

sam.to(device=DEVICE)

predictor = SamPredictor(sam)

print("SAM loaded")

SAM loaded


In [6]:
class GroundedSAMPipeline:

    def __init__(
        self,
        processor,
        model,
        sam_predictor,
        device
    ):

        self.processor = processor
        self.model = model
        self.predictor = sam_predictor
        self.device = device

    def detect(
        self,
        image_pil,
        text_prompt,
        box_threshold=0.25,
        text_threshold=0.25
    ):

        inputs = self.processor(
            images=image_pil,
            text=text_prompt,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():

            outputs = self.model(**inputs)

        results = self.processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=box_threshold,
            text_threshold=text_threshold,
            target_sizes=[image_pil.size[::-1]]
        )

        result = results[0]

        return (
            result["boxes"].cpu().numpy(),
            result["scores"].cpu().numpy(),
            result["text_labels"]
        )

    def segment(
        self,
        image_np,
        boxes
    ):

        self.predictor.set_image(image_np)

        masks_list = []

        for box in boxes:

            mask, _, _ = self.predictor.predict(
                point_coords=None,
                point_labels=None,
                box=box.astype(np.float32),
                multimask_output=False
            )

            masks_list.append(mask[0])

        return np.stack(masks_list, axis=0)

    def run(
        self,
        image_pil,
        text_prompt,
        box_threshold=0.25
    ):

        boxes, scores, labels = self.detect(
            image_pil,
            text_prompt,
            box_threshold
        )

        image_np = np.array(
            image_pil.convert("RGB")
        )

        masks = self.segment(
            image_np,
            boxes
        )

        return {
            "image_np": image_np,
            "boxes": boxes,
            "scores": scores,
            "labels": labels,
            "masks": masks
        }

In [7]:
pipeline = GroundedSAMPipeline(
    processor,
    grounding_model,
    predictor,
    DEVICE
)

print("Pipeline ready")

Pipeline ready


In [8]:
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"

image_pil = Image.open(
    requests.get(
        image_url,
        stream=True
    ).raw
).convert("RGB")

print(image_pil.size)

(640, 480)


In [9]:
text_prompt = "cat . remote control . blanket"

result = pipeline.run(
    image_pil,
    text_prompt,
    box_threshold=0.25
)

print(f"Detected {len(result['labels'])} objects")

for label, score in zip(
    result["labels"],
    result["scores"]
):
    print(label, float(score))

Detected 4 objects
cat 0.8362811207771301
cat 0.8066186904907227
remote control 0.4852662682533264
remote control 0.5202921628952026


In [10]:
def compute_structural_score(mask):

    mask_uint8 = mask.astype(np.uint8)

    area = np.sum(mask_uint8)

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        return 0.0

    perimeter = cv2.arcLength(
        contours[0],
        True
    )

    compactness = (
        4 * np.pi * area
    ) / (perimeter ** 2 + 1e-6)

    compactness = min(
        1.0,
        compactness
    )

    return float(compactness)

In [11]:
semantic_scores = result["scores"]

structural_scores = []

for mask in result["masks"]:

    structural = compute_structural_score(
        mask
    )

    structural_scores.append(
        structural
    )

In [16]:
arbitration_scores = []

for semantic, structural in zip(
    semantic_scores,
    structural_scores
):

    arbitration = (
        0.6 * float(semantic)
        +
        0.4 * structural
    )

    arbitration_scores.append(
        arbitration
    )

In [17]:
for label, semantic, structural, arbitration in zip(

    result["labels"],
    semantic_scores,
    structural_scores,
    arbitration_scores
):

    print(
        f"{label:15} | "
        f"Semantic={float(semantic):.3f} | "
        f"Structural={structural:.3f} | "
        f"Arbitrated={arbitration:.3f}"
    )

cat             | Semantic=0.836 | Structural=1.000 | Arbitrated=0.902
cat             | Semantic=0.807 | Structural=0.305 | Arbitrated=0.606
remote control  | Semantic=0.485 | Structural=0.380 | Arbitrated=0.443
remote control  | Semantic=0.520 | Structural=0.510 | Arbitrated=0.516


In [18]:
filtered_masks = []

for mask, arbitration in zip(
    result["masks"],
    arbitration_scores
):

    if arbitration > 0.5:

        filtered_masks.append(mask)

In [19]:
import os
import cv2

os.makedirs(
    "../outputs/day2",
    exist_ok=True
)

overlay = result["image_np"].copy()

colors = [
    [255, 0, 0],
    [0, 255, 0],
    [0, 0, 255],
    [255, 255, 0]
]

for i, mask in enumerate(filtered_masks):

    overlay[mask > 0] = colors[i % len(colors)]

overlay_bgr = cv2.cvtColor(
    overlay,
    cv2.COLOR_RGB2BGR
)

cv2.imwrite(
    "../outputs/day2/arbitration_filtered2.jpg",
    overlay_bgr
)

print("Saved visualization")

Saved visualization
